# CE541E08 — Unit 3 · Day 21 — Indexing and Slicing

| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 3 — The NumPy Library |
| **Session** | Day 21 of 45 |
| **CO** | CO3, CO4 |
| **Topics** | 1-D indexing · negative indexing · slicing · 2-D slicing · fancy indexing · boolean indexing |

---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge at the end.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 21"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — Indexing and Slicing in NumPy

Indexing and slicing let you **pull out exactly the part of an array you need** — a single value, a row, a column, a sub-matrix, or a scattered set of elements — without loops.

### Quick reference

| Syntax | What it selects |
|---|---|
| `arr[i]` | Single element at position i (0-based) |
| `arr[-1]` | Last element |
| `arr[i:j]` | Elements from index i up to (not including) j |
| `arr[i:]` | From index i to the end |
| `arr[:j]` | From the start up to (not including) j |
| `arr[[2,5,8]]` | Elements at positions 2, 5, and 8 (fancy indexing) |
| `arr[arr > x]` | All elements satisfying a condition (boolean indexing) |

For 2-D arrays, two indices are needed — one for rows, one for columns:

| Syntax | What it selects |
|---|---|
| `data[r, c]` | Single element at row r, column c |
| `data[:, c]` | Entire column c (all rows) |
| `data[r, :]` | Entire row r (all columns) |
| `data[r1:r2, c1:c2]` | Sub-matrix from rows r1 to r2, cols c1 to c2 |

All of this replaces what would otherwise require nested for loops.

---
## Code Block 1 — 1-D Indexing on Monthly Rainfall

### What this code does

We index into a 1-D array of 12 monthly rainfall values to extract individual months, seasonal sub-arrays, and seasonal totals — using position indexing, negative indexing, and slicing.

### Why each step is taken

**`monthly[0]` — January:**
NumPy arrays are 0-indexed, so the first element is at index 0. January is index 0, December is index 11.

**`monthly[-1]` — December:**
Negative indexing counts from the end. `-1` is always the last element, regardless of array length. This is safer than writing `monthly[11]` because it still works if the array length changes.

**`monthly[5:9]` — Monsoon (Jun–Sep):**
Slicing uses `start:stop` where stop is **not included**. June = index 5, September = index 8. So `5:9` gives indices 5, 6, 7, 8 — exactly June, July, August, September. This is the Southwest Monsoon period for the Cauvery basin.

**`monthly[[0,1]]`, `monthly[[2,3,4]]` — Fancy indexing:**
Passing a list of indices lets you select non-contiguous elements. This is called fancy indexing. It is used here to select seasonal groupings that are not contiguous slices — for example, winter months are just Jan and Feb (indices 0 and 1).

**Computing seasonal percentages:**
Dividing each seasonal total by the annual total and multiplying by 100 gives the percentage contribution. This is a standard way to characterise the seasonality of a rainfall regime.

### Algorithm

```
1. Create monthly rainfall array — 12 values (Jan to Dec)

2. Access single elements:
   monthly[0]   → January (first element)
   monthly[-1]  → December (last element, negative index)

3. Slice seasonal sub-arrays:
   monthly[5:9] → Jun, Jul, Aug, Sep (SW Monsoon)
   monthly[2:5] → Mar, Apr, May (Pre-monsoon)

4. Fancy indexing for non-contiguous seasons:
   monthly[[0,1]]      → Jan, Feb (Winter)
   monthly[[2,3,4]]    → Mar, Apr, May
   monthly[[5,6,7,8]]  → Jun–Sep
   monthly[[9,10,11]]  → Oct, Nov, Dec

5. For each season:
   .sum() → seasonal total
   / monthly.sum() * 100 → percentage of annual total
```

### Expected output

```
Jan (0)     : 8 mm
Dec (-1)    : 13 mm
Monsoon     : [134 118 113  95]  total=460 mm
Pre-monsoon : [18 52 87]  total=157 mm
  Winter          :    20 mm (2.6%)
  Pre-monsoon     :   157 mm (20.5%)
  SW Monsoon      :   460 mm (60.1%)
  Post-monsoon    :   128 mm (16.7%)
```

In [ ]:
import numpy as np

# 12 monthly rainfall totals (mm) — Cauvery basin long-term mean
# Index:  0   1   2   3   4    5    6    7   8   9  10  11
# Month: Jan Feb Mar Apr May  Jun  Jul  Aug Sep Oct Nov Dec
monthly = np.array([8, 12, 18, 52, 87, 134, 118, 113, 95, 71, 44, 13])

# Single element access — 0-based indexing
print(f"Jan (0)     : {monthly[0]} mm")    # index 0 = first element
print(f"Dec (-1)    : {monthly[-1]} mm")   # index -1 = last element

# Slicing: start:stop — stop index is NOT included
# June = index 5, September = index 8 → slice [5:9] gives indices 5,6,7,8
monsoon = monthly[5:9]
print(f"Monsoon     : {monsoon}  total={monsoon.sum()} mm")

pre = monthly[2:5]    # Mar(2), Apr(3), May(4) → indices 2,3,4
print(f"Pre-monsoon : {pre}  total={pre.sum()} mm")

# Fancy indexing with a list of indices — for non-contiguous groupings
# Seasonal breakdown with percentage of annual total
annual_total = monthly.sum()
for season, idx in [
    ('Winter',       [0, 1]),
    ('Pre-monsoon',  [2, 3, 4]),
    ('SW Monsoon',   [5, 6, 7, 8]),
    ('Post-monsoon', [9, 10, 11])
]:
    t = monthly[idx].sum()    # fancy indexing: select elements at these positions
    print(f"  {season:<16}: {t:>5} mm ({t/annual_total*100:.1f}%)")

### 🔁 Try this

Use fancy indexing to extract only the three peak monsoon months (Jul, Aug, Sep — indices 6, 7, 8).

- What is their combined total?
- What percentage of the annual total do just these three months contribute?
- Try `monthly[[6,7,8]].sum() / monthly.sum() * 100`

---
## Code Block 2 — 2-D Slicing

### What this code does

We work with a 5-year × 12-month rainfall matrix and extract individual cells, entire columns, and rectangular sub-matrices using 2-D indexing and slicing.

### Why each step is taken

**`data[1, 5]` — single cell:**
Row 1 = Year 2 (2023), Column 5 = June (0-based). This reads a single value from the matrix — the June rainfall for 2023.

**`data[:, 7]` — entire column:**
`:` means "all rows". Column 7 = August. So `data[:, 7]` extracts August rainfall for all 5 years. This is how you build a time series for one specific month.

**`.mean()` on the extracted column:**
After extracting a column as a 1-D array, all standard array methods apply. `.mean()` gives the 5-year mean August rainfall — the climatological value for that month.

**`data[:3, 5:9]` — rectangular sub-matrix:**
`data[:3]` selects the first 3 rows (years 2022, 2023, 2024). `data[5:9]` across the column axis selects June (5), July (6), August (7), September (8). Together they extract the monsoon season for the first 3 years as a `(3, 4)` sub-matrix.

### Algorithm

```
1. Create 5×12 rainfall matrix — rows = years, cols = months

2. Single element: data[row, col]
   data[1, 5] → row 1 (Year 2), col 5 (June)

3. Full column: data[:, col]
   data[:, 7] → all 5 years of August values
   .mean() → 5-year mean for August

4. Sub-matrix: data[row_start:row_stop, col_start:col_stop]
   data[:3, 5:9] → rows 0,1,2 (first 3 years) × cols 5,6,7,8 (Jun-Sep)
   Result shape: (3, 4)
   .mean() → overall mean across this sub-matrix
```

### Expected output

```
Shape: (5, 12)
Year2, June: 92 mm
Aug all years: [113 108 119 115 116]
Aug mean: 114.2 mm
Monsoon years 1-3: [[ 87 134 118 113]
                    [ 92 145 123 108]
                    [ 83 128 115 119]]
Monsoon mean: 114.7 mm
```

In [ ]:
import numpy as np

# 5-year × 12-month rainfall matrix (mm)
# Rows  = years  (Year 1 = 2022 ... Year 5 = 2026)
# Cols  = months (Col 0 = Jan ... Col 11 = Dec)
data = np.array([
    [8, 12, 18, 52, 87, 134, 118, 113, 95, 71, 44, 13],   # Year 1
    [5,  8, 22, 48, 92, 145, 123, 108, 89, 68, 38, 10],   # Year 2
    [10,15, 16, 55, 83, 128, 115, 119, 98, 74, 48, 16],   # Year 3
    [7, 10, 20, 50, 88, 138, 120, 115, 92, 70, 42, 12],   # Year 4
    [9, 14, 19, 53, 90, 140, 117, 116, 96, 72, 45, 14],   # Year 5
])
print(f"Shape: {data.shape}")

# Single element: data[row, col]
# Row 1 = Year 2 (0-based), Col 5 = June (Jan=0, Jun=5)
print(f"Year2, June: {data[1, 5]} mm")

# Full column: data[:, col]
# : in row position means all rows; col 7 = August
print(f"Aug all years: {data[:, 7]}")
print(f"Aug mean: {data[:, 7].mean():.1f} mm")

# Sub-matrix: data[row_slice, col_slice]
# data[:3]  → rows 0, 1, 2 = first 3 years
# data[5:9] → cols 5, 6, 7, 8 = Jun, Jul, Aug, Sep
monsoon_data = data[:3, 5:9]
print(f"Monsoon years 1-3: {monsoon_data}")
print(f"Monsoon mean: {monsoon_data.mean():.1f} mm")

### 🔁 Try this

Extract the **post-monsoon season** (Oct, Nov, Dec — columns 9, 10, 11) for **all 5 years**.

- What is the shape of the extracted sub-matrix?
- Which year had the highest post-monsoon total?
- Try `data[:, 9:12].sum(axis=1)` and then `.argmax() + 1` for the year number.

---
## Code Block 3 — Fancy Indexing: Catchment Analysis

### What this code does

We work with a 6-catchment × 12-month rainfall matrix and use fancy indexing to compute annual totals, monsoon totals, and identify the wettest catchment — all without loops.

### Why each step is taken

**`rainfall.sum(axis=1)` — annual totals:**
Summing across the 12 months (axis=1) gives one annual total per catchment row. This is the same axis operation from Day 20, now applied to a real basin dataset.

**`rainfall[:, 5:9]` — monsoon columns:**
Slices columns 5 to 8 (June to September) for all 6 catchments. The result is a `(6, 4)` sub-matrix — 6 catchments × 4 monsoon months.

**`.sum(axis=1)` on the monsoon sub-matrix:**
Summing across the 4 monsoon months gives one monsoon total per catchment — a `(6,)` array.

**`.argmax()` — wettest catchment:**
Returns the index (0-based) of the catchment with the highest monsoon total. We use this index to look up the name from the `catchment_names` list.

### Algorithm

```
1. Create 6×12 rainfall matrix
   rows = catchments, cols = months

2. rainfall.sum(axis=1)
   → sums 12 months for each catchment
   → shape (6,) — one annual total per catchment

3. rainfall[:, 5:9]
   → monsoon sub-matrix: all 6 catchments, cols Jun-Sep
   → shape (6, 4)

4. .sum(axis=1) on monsoon sub-matrix
   → shape (6,) — one monsoon total per catchment

5. .argmax()
   → index of maximum value (0-based)
   → use as index into catchment_names list
```

### Expected output

```
Shape: (6, 12)
  Hemavathi      : 690 mm/yr
  Harangi        : 765 mm/yr
  Kabini         : 817 mm/yr
  Suvarnavathi   : 605 mm/yr
  Shimsha        : 648 mm/yr
  Arkavathi      : 557 mm/yr
Wettest monsoon: Kabini (480 mm)
```

In [ ]:
import numpy as np

catchment_names = ['Hemavathi','Harangi','Kabini','Suvarnavathi','Shimsha','Arkavathi']

# 6-catchment × 12-month rainfall matrix (mm)
# Rows = catchments, Columns = months (Jan to Dec)
rainfall = np.array([
    [6, 10,15,45,80,120,105,108,88,65,38,10],   # Hemavathi
    [8, 12,18,52,87,134,118,113,95,71,44,13],   # Harangi
    [10,14,20,58,92,140,125,120,100,75,48,15],  # Kabini
    [4,  8,12,35,70,110, 95, 98, 78,55,32, 8],  # Suvarnavathi
    [5,  9,14,40,75,115,100,103, 83,60,35, 9],  # Shimsha
    [3,  7,11,32,65,105, 90, 93, 73,50,28, 7],  # Arkavathi
])
print(f"Shape: {rainfall.shape}")

# Annual total per catchment — sum across all 12 months (axis=1)
annual = rainfall.sum(axis=1)
for n, t in zip(catchment_names, annual):
    print(f"  {n:<15}: {t} mm/yr")

# Monsoon total per catchment
# Step 1: rainfall[:, 5:9] → select Jun-Sep columns for all catchments
# Step 2: .sum(axis=1) → sum the 4 monsoon months for each catchment
monsoon_totals = rainfall[:, 5:9].sum(axis=1)

# .argmax() → index of the catchment with the highest monsoon total
w = monsoon_totals.argmax()
print(f"Wettest monsoon: {catchment_names[w]} ({monsoon_totals[w]} mm)")

### 🔁 Try this

Find the **driest catchment** in the monsoon season using `.argmin()`.

- Which catchment is it?
- What is its monsoon total compared to the wettest?
- Also try: which catchment has the highest annual total? Use `annual.argmax()`.

---
## Code Block 4 — Boolean Indexing for Quality Control

### What this code does

We apply boolean indexing to a streamflow record to identify flood days, extract only the flood flow values, and compute the mean flow during normal (non-flood) conditions — all in three lines.

### Why each step is taken

**`np.where(flow > 600)[0]`:**
`flow > 600` produces a True/False array. `np.where(condition)` returns a **tuple** containing the indices where the condition is True. We take `[0]` to get the array of indices from that tuple. Adding `+1` converts from 0-based indices to 1-based day numbers.

**`flow[flow > 600]`:**
Boolean indexing directly selects the values — no need to know the indices. This gives the actual discharge values on flood days, not their positions.

**`flow[flow <= 600].mean()`:**
The opposite condition selects only normal-flow days. `.mean()` on this filtered array gives the baseline flow — what the river looks like when it is not flooding.

**Why this matters:**
In hydrology, separating flood events from baseflow is a fundamental step in flood frequency analysis and reservoir operations planning. Without NumPy, this would require a loop, a counter, and a running total.

### Algorithm

```
1. Create streamflow array (15 daily values, m³/s)

2. np.where(flow > 600)[0]
   → flow > 600 produces True/False array
   → np.where returns tuple of indices where True
   → [0] extracts the index array
   → +1 converts to 1-based day numbers

3. flow[flow > 600]
   → boolean indexing selects flood-day values directly
   → no index needed — just the condition

4. flow[flow <= 600].mean()
   → selects non-flood days
   → .mean() gives baseline (normal) flow
```

### Expected output

```
Flood days   : [ 4  5  6  7  8]
Flood flows  : [ 890 1245  987  756  543]
Normal mean  : 284.4 m3/s
```

In [ ]:
import numpy as np

# 15-day streamflow record (m³/s) — Cauvery at KRS, July 2024
# Day:    1    2    3    4     5    6    7    8    9   10   11   12   13   14   15
flow = np.array([234,267,312,890,1245,987,756,543,412,345,289,245,212,198,220])

# np.where(condition) returns a TUPLE of arrays — one per dimension
# For 1-D arrays, the tuple has one element: the array of indices where True
# [0] extracts that index array; +1 converts 0-based to 1-based day numbers
flood_days = np.where(flow > 600)[0] + 1
print(f"Flood days   : {flood_days}")

# Boolean indexing: flow > 600 → True/False array
# flow[flow > 600] selects the actual values where True
print(f"Flood flows  : {flow[flow > 600]}")

# Opposite condition: flow <= 600 selects normal (non-flood) days
# .mean() on the filtered array gives the baseline flow
print(f"Normal mean  : {flow[flow <= 600].mean():.1f} m3/s")

### 🔁 Try this

Change the flood threshold from `600` to `400` m³/s.

- How many flood days are there now?
- What is the new normal-flow mean?
- Does this seem like a reasonable threshold for this river? Why or why not?

---
## Session Summary — Indexing and Slicing in NumPy

| Operation | Syntax | Example result |
|---|---|---|
| Single element (1-D) | `arr[i]` | `monthly[5]` → June value |
| Last element | `arr[-1]` | `monthly[-1]` → December |
| Slice (1-D) | `arr[start:stop]` | `monthly[5:9]` → Jun–Sep |
| Fancy indexing | `arr[[i,j,k]]` | `monthly[[0,1]]` → Jan, Feb |
| Single element (2-D) | `data[row, col]` | `data[1,5]` → Year2 June |
| Full column | `data[:, col]` | `data[:,7]` → all years, August |
| Full row | `data[row, :]` | `data[2,:]` → Year3, all months |
| Sub-matrix | `data[r1:r2, c1:c2]` | `data[:3,5:9]` → first 3 yrs, monsoon |
| Boolean select | `arr[arr > x]` | `flow[flow>600]` → flood values |
| Boolean indices | `np.where(arr>x)[0]` | flood day numbers |
| Count matching | `(arr > x).sum()` | number of flood days |

---
## Day 21 Assignment

5-station × 12-month streamflow matrix (m³/s):

```python
flows = np.array([
    [12, 8, 6, 15, 45, 234, 456, 378, 189, 67, 28, 14],
    [8,  5, 4, 10, 32, 178, 345, 289, 145, 48, 20,  9],
    [18,12, 9, 22, 60, 312, 567, 489, 245, 89, 38, 18],
    [6,  4, 3,  8, 25, 134, 267, 213, 108, 38, 16,  7],
    [15,10, 7, 18, 50, 256, 489, 401, 200, 72, 31, 14],
])
```

1. Extract the monsoon sub-matrix (Jun–Sep) for all 5 stations
2. Find which station has the highest total monsoon flow using `argmax()`
3. Extract the full annual record for Station 3 (row index 2)
4. Find all months across all stations where flow exceeded 300 m³/s — use `np.where()`

### ▶ Assignment cell

In [ ]:
import numpy as np

flows = np.array([
    [12, 8,  6, 15, 45, 234, 456, 378, 189, 67, 28, 14],
    [8,  5,  4, 10, 32, 178, 345, 289, 145, 48, 20,  9],
    [18,12,  9, 22, 60, 312, 567, 489, 245, 89, 38, 18],
    [6,  4,  3,  8, 25, 134, 267, 213, 108, 38, 16,  7],
    [15,10,  7, 18, 50, 256, 489, 401, 200, 72, 31, 14],
])

monsoon      = ???    # monsoon sub-matrix: all stations, Jun-Sep
best         = ???    # index of station with highest monsoon total
station3     = ???    # full record for station 3 (row index 2)
high_months  = ???    # np.where(flows > 300) — returns (row_indices, col_indices)

print(f"Best station (monsoon): Station {best + 1}")
print(f"Station 3 annual      : {station3}")
print(f"High flow locations   : rows={high_months[0]+1}, cols={high_months[1]+1}")

---
- [ ] Run all cells from top to bottom — verify your outputs match the expected outputs above
- [ ] Complete the assignment cell (replace `???` placeholders)
- [ ] Upload to GitHub: `Unit3_NumPy/CE541E08_U3_Day21.ipynb`
- [ ] Commit message: `Day 21 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*